In [0]:
tables = ["cost_centers", "vendor_master", "gl_entries"]

for table_name in tables:
    volume_path = f"/Volumes/zaitoon_catalog/bronze/raw_events/sap_hana/{table_name}"
    checkpoint_path = f"/Volumes/zaitoon_catalog/bronze/raw_events/_checkpoints/sap_{table_name}"
    schema_path = f"/Volumes/zaitoon_catalog/bronze/raw_events/_schemas/sap_{table_name}"
    bronze_table = f"zaitoon_catalog.bronze.sap_{table_name}"

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("multiLine", "true")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .load(volume_path)
    )

    (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .toTable(bronze_table)
    )

    print(f"Loaded {table_name} into {bronze_table}")

In [0]:
for table_name in tables:
    count = spark.table(f"zaitoon_catalog.bronze.sap_{table_name}").count()
    print(f"sap_{table_name}: {count} rows")

In [0]:
display(spark.read.option("multiLine", "true").json("/Volumes/zaitoon_catalog/bronze/raw_events/sap_hana/gl_entries"))